In [1]:
import psycopg2

try:
    conn = psycopg2.connect(
        dbname="testdb",
        user="postgres",
        password="Ok1234567890.",  # Bytt ut med ditt passord hvis det er annerledes
        host="localhost",
        port="5432"
    )
    
    cur = conn.cursor()
    print("Tilkobling vellykket! 🟢")

    cur.execute("SELECT * FROM users_data;")
    rows = cur.fetchall()
    print(f"Antall brukere i tabellen akkurat nå: {len(rows)}")

    cur.close()
    conn.close()

except Exception as e:
    print(f"Feilet ved tilkobling: {e}")

Tilkobling vellykket! 🟢
Antall brukere i tabellen akkurat nå: 0


In [ ]:
# test av database

In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
import psycopg2

app = FastAPI()

# Funksjon for å koble til den lokale databasen vi lagde i Steg 1
def get_connection():
    return psycopg2.connect(
        dbname="testdb",
        user="postgres",
        password="Ok1234567890.",  # Bytt ut hvis du har et annet passord
        host="localhost",
        port="5432"
    )

# Pydantic-skjema for dataene som kommer inn
class UserData(BaseModel):
    name: str
    email: str
    message: str = None

# GET-endepunkt: Henter alle brukere fra databasen
@app.get("/users")
def get_users():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users_data")
    rows = cursor.fetchall()
    cursor.close()
    conn.close()
    return {"users": rows}

# POST-endepunkt: Legger til en ny bruker i databasen
@app.post("/users")
def add_user(user: UserData):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO users_data (name, email, message) VALUES (%s, %s, %s)",
        (user.name, user.email, user.message)
    )
    conn.commit()
    cursor.close()
    conn.close()
    return {"message": "User added successfully!", "data": user}

# Lager en klient for å teste API-et internt i notebooken
client = TestClient(app)
print("FastAPI satt opp og klar til testing! 🟢")

FastAPI satt opp og klar til testing! 🟢


In [2]:
response = client.post("/users", json={
    "name": "Kari Nordmann",
    "email": "kari@eksempel.no",
    "message": "Dette er en testmelding fra Jupyter!"
})

print("Statuskode:", response.status_code)
print("Svar fra API:", response.json())

Statuskode: 200
Svar fra API: {'message': 'User added successfully!', 'data': {'name': 'Kari Nordmann', 'email': 'kari@eksempel.no', 'message': 'Dette er en testmelding fra Jupyter!'}}


In [3]:
response = client.get("/users")
print("Statuskode:", response.status_code)
print("Alle brukere i databasen:", response.json())

Statuskode: 200
Alle brukere i databasen: {'users': [[1, 'Kari Nordmann', 'kari@eksempel.no', 'Dette er en testmelding fra Jupyter!']]}


In [ ]:
# Sette dataen i backend

In [ ]:
# Lage index.html fil. 
# I terminal sette uvicorn main:app --reload
# Klikk på live server og start. 

In [4]:
import psycopg2
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

# 1. Test ren databasetilkobling først
try:
    conn = psycopg2.connect(
        dbname="db",
        user="postgres",
        password="Ok1234567890.",
        host="localhost",
        port="5432"
    )
    
    cur = conn.cursor()
    print("Tilkobling til 'db' vellykket! 🟢")

    cur.execute("SELECT * FROM kontakter;")
    rows = cur.fetchall()
    print(f"Antall rader i 'kontakter' før test: {len(rows)}")

    cur.close()
    conn.close()

except Exception as e:
    print(f"Feilet ved tilkobling: {e}")

# 2. Sett opp FastAPI-appen mot den nye databasen og tabellen
app = FastAPI()

def get_connection():
    return psycopg2.connect(
        dbname="db",
        user="postgres",
        password="Ok1234567890.",
        host="localhost",
        port="5432"
    )

class UserData(BaseModel):
    name: str
    email: str
    message: str | None = None

@app.get("/users")
def get_users():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT id, name, email, message FROM kontakter")
    rows = cursor.fetchall()
    cursor.close()
    conn.close()
    return {"users": rows}

@app.post("/users")
def add_user(user: UserData):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO kontakter (name, email, message) VALUES (%s, %s, %s)",
        (user.name, user.email, user.message)
    )
    conn.commit()
    cursor.close()
    conn.close()
    return {"message": "Lagret i kontakter!", "data": user}

# 3. Kjør intern API-test med TestClient
client = TestClient(app)
print("\nFastAPI satt opp og klar til testing! 🟢")

# Send en testkontakt (POST)
post_response = client.post("/users", json={
    "name": "Kari Nordmann",
    "email": "kari@eksempel.no",
    "message": "Test fra Jupyter til ny db og tabell!"
})
print("POST statuskode:", post_response.status_code)
print("POST svar:", post_response.json())

# Hent alle kontakter for å sjekke at den ble lagret (GET)
get_response = client.get("/users")
print("\nGET statuskode:", get_response.status_code)
print("Alle rader i 'kontakter' nå:", get_response.json())

Tilkobling til 'db' vellykket! 🟢
Antall rader i 'kontakter' før test: 0

FastAPI satt opp og klar til testing! 🟢
POST statuskode: 200
POST svar: {'message': 'Lagret i kontakter!', 'data': {'name': 'Kari Nordmann', 'email': 'kari@eksempel.no', 'message': 'Test fra Jupyter til ny db og tabell!'}}

GET statuskode: 200
Alle rader i 'kontakter' nå: {'users': [[1, 'Kari Nordmann', 'kari@eksempel.no', 'Test fra Jupyter til ny db og tabell!']]}


In [5]:
# Teste databasen

In [6]:
import psycopg2

try:
    conn = psycopg2.connect(
        dbname="kontakter_db",
        user="kontakter_db_user",
        password="CDoDdCKvRZPWGmHEOwG4Bvg7MqnkQ4et",
        host="dpg-dc29rahksfps73dh38q0-a.frankfurt-postgres.render.com",
        port="5432",
        sslmode="require"
    )
    
    cur = conn.cursor()
    print("Tilkobling til Render-databasen vellykket! 🟢")

    cur.execute("SELECT * FROM kontakter;")
    rows = cur.fetchall()
    print(f"Antall rader i 'kontakter' akkurat nå: {len(rows)}")

    cur.close()
    conn.close()

except Exception as e:
    print(f"Feilet ved tilkobling: {e}")

Tilkobling til Render-databasen vellykket! 🟢
Antall rader i 'kontakter' akkurat nå: 0


In [7]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
import psycopg2

app = FastAPI()

def get_connection():
    return psycopg2.connect(
        dbname="kontakter_db",
        user="kontakter_db_user",
        password="CDoDdCKvRZPWGmHEOwG4Bvg7MqnkQ4et",
        host="dpg-dc29rahksfps73dh38q0-a.frankfurt-postgres.render.com",
        port="5432",
        sslmode="require"
    )

class UserData(BaseModel):
    name: str
    email: str
    message: str | None = None

@app.get("/users")
def get_users():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT id, name, email, message FROM kontakter ORDER BY id ASC")
    rows = cursor.fetchall()
    cursor.close()
    conn.close()
    return {"users": rows}

@app.post("/users")
def add_user(user: UserData):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO kontakter (name, email, message) VALUES (%s, %s, %s)",
        (user.name, user.email, user.message)
    )
    conn.commit()
    cursor.close()
    conn.close()
    return {"message": "Lagret i Render-databasen!", "data": user}

@app.delete("/users/{user_id}")
def delete_user(user_id: int):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM kontakter WHERE id = %s", (user_id,))
    conn.commit()
    cursor.close()
    conn.close()
    return {"message": f"Kontakt med ID {user_id} ble slettet!"}

# --- Kjør tester mot Render-databasen ---
client = TestClient(app)
print("FastAPI klar til testing mot Render-skyen! 🟢")

# 1. Test POST (legge til rad)
post_res = client.post("/users", json={
    "name": "Ola Nordmann",
    "email": "ola@render.no",
    "message": "Test fra Jupyter rett inn i Render PostgreSQL!"
})
print("POST status:", post_res.status_code, "| Svar:", post_res.json())

# 2. Test GET (hente rader)
get_res = client.get("/users")
print("GET status:", get_res.status_code, "| Innhold i databasen:", get_res.json())

FastAPI klar til testing mot Render-skyen! 🟢
POST status: 200 | Svar: {'message': 'Lagret i Render-databasen!', 'data': {'name': 'Ola Nordmann', 'email': 'ola@render.no', 'message': 'Test fra Jupyter rett inn i Render PostgreSQL!'}}
GET status: 200 | Innhold i databasen: {'users': [[1, 'Ola Nordmann', 'ola@render.no', 'Test fra Jupyter rett inn i Render PostgreSQL!']]}
